In [ ]:
"""
Temporal Holdout Backtesting for Global-Level Uncertainty Estimation
====================================================================
Train TabPFN models at multiple time cutoffs, predict forward,
aggregate to global totals, and compute WAPE at each forecast horizon.

Output: 3_temporal_backtest_results.csv  (per cutoff × year × waste type)
        3_global_error_vs_horizon.csv    (aggregated error curve)
"""

from __future__ import annotations
from pathlib import Path
from contextlib import contextmanager
import gc

import numpy as np
import pandas as pd
import torch
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

# ── Configuration ──
CUTOFF_YEARS = [2005, 2010, 2015, 2019]
WASTE_TYPES = ['AW', 'CDW', 'IW', 'MSW']
FEATURE_CONTRACT = [
    'NY.GDP.MKTP.PP.KD',
    'SP.POP.TOTL',
    'NY.GDP.PCAP.PP.KD',
    'EN.POP.DNST',
    'Proxy_NY.GDP.MKTP.PP.KD_log1p',
    'Proxy_SP.POP.TOTL_log1p',
    'Proxy_NY.GDP.PCAP.PP.KD_log1p',
    'Proxy_EN.POP.DNST_log1p',
    'EN.GHG.CO2.MT.CE.AR5',
    'Proxy_EN.GHG.CO2.MT.CE.AR5_log1p',
]
TARGET_COL = 'Label_Log'
TABPFN_N_ESTIMATORS = 8
TABPFN_SUBSAMPLE_SAMPLES = 2000
SEED = 42
BATCH_SIZE = 512

cwd = Path.cwd().resolve()

In [ ]:
def find_release_root(start: Path) -> Path:
    sentinel_dirs = ('Step0_Data', 'Step9_FutureProjection')
    for candidate in [start, *start.parents]:
        if all((candidate / name).is_dir() for name in sentinel_dirs):
            return candidate
        nested_project = candidate / 'Global-solid-waste-prediction'
        if all((nested_project / name).is_dir() for name in sentinel_dirs):
            return nested_project
    raise FileNotFoundError('Could not locate Global-solid-waste-prediction repository root')


release_root = find_release_root(cwd)
step6_results = release_root / 'Step7_ModelTraining' / '2_Results'
step8_results = release_root / 'Step9_FutureProjection' / '2_Results'
step8_results.mkdir(parents=True, exist_ok=True)

# ── Load Step6 preprocessing pipelines and processed feature contract ──
processed_contract = pd.read_csv(
    step6_results / '0_feature_contract_processed.csv'
)['feature'].astype(str).tolist()

print(f'Release root: {release_root}')
print(f'Processed feature contract: {len(processed_contract)} features')

In [ ]:
def resolve_model_version():
    if hasattr(ModelVersion, 'V2_6'):
        return getattr(ModelVersion, 'V2_6')
    if hasattr(ModelVersion, 'V2_5'):
        return ModelVersion.V2_5
    return ModelVersion.V2


def predict_in_batches(model, features, batch_size=512):
    n_rows = len(features)
    if batch_size is None or batch_size <= 0 or batch_size >= n_rows:
        return np.asarray(model.predict(features), dtype=float).reshape(-1)
    predictions = []
    n_batches = (n_rows - 1) // batch_size + 1
    for batch_idx in range(n_batches):
        start = batch_idx * batch_size
        end = min((batch_idx + 1) * batch_size, n_rows)
        batch_features = features.iloc[start:end]
        batch_pred = np.asarray(model.predict(batch_features), dtype=float).reshape(-1)
        predictions.append(batch_pred)
    return np.concatenate(predictions, axis=0)


model_version = resolve_model_version()
print(f'TabPFN version: {model_version.value}')

In [ ]:
# ── Load training data (reuse Step6 processed data) ──
all_backtest_results = []

for waste_type in WASTE_TYPES:
    # Load the Step6 already-processed training data (PyCaret transforms already applied)
    train_processed = pd.read_csv(step6_results / f'0_training_processed_{waste_type}.csv')

    n_total = len(train_processed)
    print(f'\n{"="*60}')
    print(f'{waste_type}: {n_total} training rows, years {int(train_processed["Year"].min())}-{int(train_processed["Year"].max())}')

    for cutoff in CUTOFF_YEARS:
        # Split: train on <= cutoff, test on > cutoff
        train_mask = train_processed['Year'] <= cutoff
        test_mask = train_processed['Year'] > cutoff

        if train_mask.sum() == 0 or test_mask.sum() == 0:
            print(f'  Cutoff {cutoff}: skip (no train or test rows)')
            continue

        x_train = train_processed.loc[train_mask, processed_contract].copy()
        y_train = train_processed.loc[train_mask, TARGET_COL].astype(float).values
        x_test = train_processed.loc[test_mask, processed_contract].copy()
        y_test_log = train_processed.loc[test_mask, TARGET_COL].astype(float).values
        test_meta = train_processed.loc[test_mask, ['Country Code', 'Year']].copy().reset_index(drop=True)

        # Train model
        model = TabPFNRegressor.create_default_for_version(
            model_version,
            n_estimators=TABPFN_N_ESTIMATORS,
            device='cuda' if torch.cuda.is_available() else 'cpu',
            fit_mode='low_memory',
            memory_saving_mode='auto',
            ignore_pretraining_limits=True,
            inference_config={'SUBSAMPLE_SAMPLES': TABPFN_SUBSAMPLE_SAMPLES},
            random_state=SEED,
        )
        print(f'  Cutoff {cutoff}: train={train_mask.sum()}, test={test_mask.sum()} ... ', end='')
        model.fit(x_train, y_train)
        preds_log = predict_in_batches(model, x_test, batch_size=BATCH_SIZE)

        # Convert to raw scale
        preds_raw = np.expm1(preds_log)
        actual_raw = np.expm1(y_test_log)

        # Store results
        result = test_meta.copy()
        result['WasteType'] = waste_type
        result['Cutoff'] = cutoff
        result['Actual_Raw'] = actual_raw
        result['Predicted_Raw'] = preds_raw
        result['Horizon'] = result['Year'] - cutoff
        all_backtest_results.append(result)

        # Quick summary
        total_actual = actual_raw.sum() / 1e9
        total_pred = preds_raw.sum() / 1e9
        print(f'done (global: actual={total_actual:.1f} Gt, pred={total_pred:.1f} Gt)')

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
# ── Combine and save raw results ──
backtest_df = pd.concat(all_backtest_results, ignore_index=True)
raw_path = step8_results / '3_temporal_backtest_results.csv'
backtest_df.to_csv(raw_path, index=False)
print(f'\nSaved raw results: {raw_path} ({len(backtest_df)} rows)')

In [ ]:
# ── Aggregate to global-level error by cutoff × year × waste type ──
annual_by_waste = backtest_df.groupby(['Cutoff', 'Year', 'WasteType', 'Horizon']).agg(
    Global_Actual=('Actual_Raw', 'sum'),
    Global_Predicted=('Predicted_Raw', 'sum'),
    N_Countries=('Country Code', 'nunique'),
).reset_index()
annual_by_waste['Error'] = annual_by_waste['Global_Predicted'] - annual_by_waste['Global_Actual']
annual_by_waste['Abs_Error'] = annual_by_waste['Error'].abs()
annual_by_waste['Rel_Error_Pct'] = annual_by_waste['Error'] / annual_by_waste['Global_Actual'] * 100

# Also compute total SW (all waste types combined)
annual_total = backtest_df.groupby(['Cutoff', 'Year', 'Horizon']).agg(
    Global_Actual=('Actual_Raw', 'sum'),
    Global_Predicted=('Predicted_Raw', 'sum'),
    N_Countries=('Country Code', 'nunique'),
).reset_index()
annual_total['WasteType'] = 'Total_SW'
annual_total['Error'] = annual_total['Global_Predicted'] - annual_total['Global_Actual']
annual_total['Abs_Error'] = annual_total['Error'].abs()
annual_total['Rel_Error_Pct'] = annual_total['Error'] / annual_total['Global_Actual'] * 100

all_annual = pd.concat([annual_by_waste, annual_total], ignore_index=True)
all_annual = all_annual.sort_values(['WasteType', 'Cutoff', 'Year']).reset_index(drop=True)

In [ ]:
# ── Aggregate error by horizon (average across cutoffs) ──
error_by_horizon = all_annual.groupby(['WasteType', 'Horizon']).agg(
    Mean_Abs_Rel_Error_Pct=('Rel_Error_Pct', lambda x: x.abs().mean()),
    Median_Abs_Rel_Error_Pct=('Rel_Error_Pct', lambda x: x.abs().median()),
    Std_Abs_Rel_Error_Pct=('Rel_Error_Pct', lambda x: x.abs().std()),
    N_Cutoffs=('Cutoff', 'nunique'),
    Mean_Signed_Error_Pct=('Rel_Error_Pct', 'mean'),
).reset_index()
error_by_horizon = error_by_horizon.sort_values(['WasteType', 'Horizon']).reset_index(drop=True)

horizon_path = step8_results / '3_global_error_vs_horizon.csv'
error_by_horizon.to_csv(horizon_path, index=False)
print(f'Saved error curve: {horizon_path}')

In [ ]:
# ── Summary table ──
print('\n' + '=' * 70)
print('GLOBAL-LEVEL ERROR BY HORIZON (Total_SW)')
print('=' * 70)
total_err = error_by_horizon[error_by_horizon['WasteType'] == 'Total_SW']
for _, row in total_err.iterrows():
    h = int(row['Horizon'])
    mae = row['Mean_Abs_Rel_Error_Pct']
    bias = row['Mean_Signed_Error_Pct']
    n = int(row['N_Cutoffs'])
    print(f'  Horizon {h:2d} yr: WAPE={mae:.2f}% (bias={bias:+.2f}%, from {n} cutoffs)')

print('\n' + '=' * 70)
print('GLOBAL-LEVEL ERROR BY HORIZON (Per Waste Type)')
print('=' * 70)
for wt in WASTE_TYPES:
    wt_err = error_by_horizon[error_by_horizon['WasteType'] == wt]
    short = wt_err[wt_err['Horizon'] <= 4]['Mean_Abs_Rel_Error_Pct'].mean()
    long_h = wt_err['Horizon'].max()
    long_e = wt_err[wt_err['Horizon'] == long_h]['Mean_Abs_Rel_Error_Pct'].values
    long_val = long_e[0] if len(long_e) > 0 else float('nan')
    print(f'  {wt}: short-term(1-4yr)={short:.2f}%, longest({long_h}yr)={long_val:.2f}%')